# Week 4: Unusual spending

Goal: find purchases (and weeks) that are far above **your usual**, so Ollie can say *"That ₱4,800 Shopping purchase is about 4× your usual."*

You'll compare three methods and choose the numbers the app ships: **multiplier**, **minimum amount**, **minimum earlier purchases**, **minimum earlier weeks**.

Spec: `docs/superpowers/specs/2026-10-22-ollie-anomaly-alerts-design.md`

## Setup
The data now has an `Anomaly` column: `spike` (one unusually big purchase), `spree` (many normal purchases in one week), or blank. These are the **answers**, so you can score each method.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_fscore_support

df = pd.read_csv('../data/fake_transactions.csv')
df['Anomaly'] = df['Anomaly'].fillna('')

# Safety check: the generator only ADDED spree rows; every older row is unchanged
before = pd.read_csv('../data/fake_transactions_before.csv')
assert before.equals(df[df['Anomaly'] != 'spree'].drop(columns='Anomaly').reset_index(drop=True)), 'old rows changed!'

df['Date'] = pd.to_datetime(df['Date'])
exp = df[df['Type'] == 'expense'].sort_values('Date', kind='stable').reset_index(drop=True)
exp['Anomaly'].value_counts()

Anomaly
         1358
spree      30
spike      13
Name: count, dtype: int64

In [3]:
# Sanity check on the labels: spikes are ~1% of expenses, and there are 5–7 spree weeks
assert 0.005 <= (exp['Anomaly'] == 'spike').mean() <= 0.02
spree_rows = exp[exp['Anomaly'] == 'spree']
assert 5 <= spree_rows.groupby(['Category', spree_rows['Date'].dt.tz_localize(None).dt.to_period('W-SUN')]).ngroups <= 7
print('labels look right ✓')

labels look right ✓


In [12]:
def score(pred):
    """Precision, recall and F1 of a True/False prediction against the 'spike' labels."""
    p, r, f, _ = precision_recall_fscore_support(exp['Anomaly'] == 'spike', pred, average='binary', zero_division=0)
    return {'precision': p, 'recall': r, 'f1': f}

---
## A. What is "usual"? (no looking ahead)
For each expense, **usual** = the median of **earlier** purchases in the same category. Using later purchases would be **leakage**: the app can't know the future.

Add three columns:
- `earlier`: how many earlier purchases in this category. Hint: `groupby('Category').cumcount()`
- `usual`: median of earlier amounts. Hint: `groupby('Category')['Amount'].transform(lambda s: s.shift().expanding().median())`
  - `shift()` drops the current row, so only earlier ones count
- `ratio`: `Amount / usual`

**Think:** why the median and not the mean? (One ₱20,000 spike would drag a mean up.)

In [15]:
# TODO
# Make sure "earlier" really means earlier in time
exp = exp.sort_values('Date').reset_index(drop=True)

# How many purchases in this category came before this one
exp['earlier'] = exp.groupby('Category').cumcount()

# Median of earlier amounts only (shift drops the current row)
exp['usual'] = (
    exp.groupby('Category')['Amount']
      .transform(lambda s: s.shift().expanding().median())
)

# How this purchase compares to what's usual
exp['ratio'] = exp['Amount'] / exp['usual']

exp[['Date', 'Category', 'Amount', 'earlier', 'usual', 'ratio']].head(10)

,Date,Category,Amount,earlier,usual,ratio
0,2025-10-21 09:37:00+00:00,Food & Drinks,73.51,0,NaN,NaN
1,2025-10-21 17:06:00+00:00,Transportation,20.75,0,NaN,NaN
2,2025-10-21 20:14:00+00:00,Coffee & Snacks,59.68,0,NaN,NaN
3,2025-10-22 08:14:00+00:00,Shopping,2957.14,0,NaN,NaN
4,2025-10-22 09:38:00+00:00,Transportation,337.88,1,20.750,16.283373
5,2025-10-22 18:36:00+00:00,Food & Drinks,127.41,1,73.510,1.733234
6,2025-10-22 19:17:00+00:00,Groceries,913.46,0,NaN,NaN
7,2025-10-23 11:15:00+00:00,Food & Drinks,447.67,2,100.460,4.456201
8,2025-10-23 15:35:00+00:00,Transportation,156.58,2,179.315,0.873212
9,2025-10-23 22:05:00+00:00,Coffee & Snacks,95.18,1,59.680,1.594839


These checks prove there's no look-ahead. If they pass, part A is right.

In [17]:
# The first purchase in each category has nothing earlier
assert exp.groupby('Category').head(1)['usual'].isna().all()
# Recompute one row by hand from earlier rows only
row = exp[exp['earlier'] == 10].iloc[0]
hand = exp[(exp['Category'] == row['Category']) & (exp.index < row.name)]['Amount'].median()
assert abs(hand - row['usual']) < 1e-9
print('no look-ahead ✓')

no look-ahead ✓


---
## B. Three methods for single purchases
Score each with `score(...)`. It returns precision, recall and F1.

- **Precision:** of the alerts, how many were real spikes? (low = annoying false alarms)
- **Recall:** of the real spikes, how many did we catch?
- **F1:** one number balancing both

### B1. Ratio rule (what the app will use)
Flag when `ratio >= 3` **and** `Amount >= 500` **and** `earlier >= 5`. Save it as `ratio_pred`.

In [18]:
# TODO
ratio_pred = (
    (exp['ratio'] >= 3) &
    (exp['Amount'] >= 500) &
    (exp['earlier'] >= 5)
)

score(ratio_pred)

{'precision': 0.45454545454545453,
 'recall': 0.7692307692307693,
 'f1': 0.5714285714285714}

### B2. Robust z-score
How many *typical spreads* above usual is it?
- Spread = **MAD** (median absolute deviation) of earlier amounts. Hint: same `transform(lambda s: s.shift().expanding()...)` pattern, with `.apply(lambda w: np.nanmedian(np.abs(w - np.nanmedian(w))), raw=True)` (use `nanmedian`: `shift()` leaves a NaN that plain `np.median` turns into NaN for every row)
- `z = (Amount - usual) / (1.4826 * mad)`. Flag `z >= 3.5` and `earlier >= 5`. Save as `z_pred`.

In [19]:
# TODO

exp['mad'] = (
    exp.groupby('Category')['Amount']
       .transform(lambda s: s.shift().expanding().apply(
           lambda w: np.nanmedian(np.abs(w - np.nanmedian(w))), raw=True))
)

exp['z'] = (exp['Amount'] - exp['usual']) / (1.4826 * exp['mad'])

z_pred = (exp['z'] >= 3.5) & (exp['earlier'] >= 5)

score(z_pred)

{'precision': 0.5625, 'recall': 0.6923076923076923, 'f1': 0.6206896551724138}

### B3. Isolation Forest (real ML)
It learns what "normal" looks like and isolates rare points.
- Features: category one-hot (`pd.get_dummies(exp['Category']).astype(float)`) plus `np.log(exp['Amount'])`
- `IsolationForest(contamination=0.01, random_state=42)`, then `fit_predict(X) == -1` means anomaly. Save as `iso_pred`.

**Think:** Isolation Forest is fitted on *all* rows, future included. Is that a fair comparison with the ratio rule?

In [20]:
# TODO
X = pd.get_dummies(exp['Category']).astype(float)
X['log_amount'] = np.log(exp['Amount'])

iso = IsolationForest(contamination=0.01, random_state=42)
iso_pred = pd.Series(iso.fit_predict(X) == -1, index=exp.index)

score(iso_pred)

{'precision': 0.07142857142857142,
 'recall': 0.07692307692307693,
 'f1': 0.07407407407407407}

### B4. Compare
Put the three scores in one table. Hint: `pd.DataFrame({'ratio rule': score(ratio_pred), ...}).T.round(2)`

In [21]:
# TODO
results = pd.DataFrame({
    'ratio rule':       score(ratio_pred),
    'robust z':         score(z_pred),
    'isolation forest': score(iso_pred),
}).T.round(2)

results

,precision,recall,f1
ratio rule,0.45,0.77,0.57
robust z,0.56,0.69,0.62
isolation forest,0.07,0.08,0.07


---
## C. Tune the ratio rule
Try multipliers `[2, 2.5, 3, 3.5, 4, 5]` × minimum amounts `[0, 250, 500, 1000]`. For each, record precision, recall, F1 and the number of alerts.

Hint: loop, build a list of dicts, `pd.DataFrame(rows)`, sort by `f1`.

**Think:** a lower multiplier catches more spikes. What does it cost the user?

**Look at the false alarms:** show the rows the rule flags that are *not* spikes, e.g. `exp[ratio_pred & (exp['Anomaly'] != 'spike')][['Date', 'Category', 'Description', 'Amount', 'usual', 'ratio']]`. Which category keeps appearing, and why? (Hint: what else lives in that category, and how much does each cost?)

In [22]:
# TODO
rows = []
for mult in [2, 2.5, 3, 3.5, 4, 5]:
    for min_amt in [0, 250, 500, 1000]:
        pred = (
            (exp['ratio'] >= mult) &
            (exp['Amount'] >= min_amt) &
            (exp['earlier'] >= 5)
        )
        rows.append({
            'mult': mult,
            'min_amount': min_amt,
            **score(pred),
            'alerts': int(pred.sum()),
        })

tuning = pd.DataFrame(rows).sort_values('f1', ascending=False).round(2)
tuning.head(10)

,mult,min_amount,precision,recall,f1,alerts
16,4.0,0,0.64,0.69,0.67,14
17,4.0,250,0.64,0.69,0.67,14
18,4.0,500,0.64,0.69,0.67,14
19,4.0,1000,0.64,0.69,0.67,14
20,5.0,0,0.88,0.54,0.67,8
21,5.0,250,0.88,0.54,0.67,8
22,5.0,500,0.88,0.54,0.67,8
23,5.0,1000,0.88,0.54,0.67,8
12,3.5,0,0.50,0.69,0.58,18
13,3.5,250,0.50,0.69,0.58,18


---
## D. Unusual weeks (sprees)
Now the total per **category per week** (weeks start Monday).

1. Week label: `exp['Date'].dt.tz_localize(None).dt.to_period('W-SUN')`
2. Weekly totals: `pivot_table(index='week', columns='Category', values='Amount', aggfunc='sum', fill_value=0)`, then reindex to **every** week so empty weeks count as ₱0
3. Per category, starting at its first spending week: `usual` = median of earlier weeks (same `shift().expanding().median()` idea), `earlier_weeks` = count, `ratio` = total / usual
4. Build one table `weeks` with columns `Category, week, total, usual, earlier_weeks, ratio`

In [23]:
# TODO
# 1. Week label (W-SUN = weeks that end on Sunday, so they start on Monday)
exp['week'] = exp['Date'].dt.tz_localize(None).dt.to_period('W-SUN')

# 2. Weekly totals per category, with every week present
weekly = exp.pivot_table(index='week', columns='Category', values='Amount',
                         aggfunc='sum', fill_value=0)
all_weeks = pd.period_range(weekly.index.min(), weekly.index.max(), freq='W-SUN')
weekly = weekly.reindex(all_weeks, fill_value=0)
weekly.index.name = 'week'

# 3. Per category, starting at its first spending week
parts = []
for cat in weekly.columns:
    s = weekly[cat]
    s = s.loc[s.ne(0).idxmax():]          # drop weeks before the category existed
    parts.append(pd.DataFrame({
        'Category': cat,
        'week': s.index,
        'total': s.values,
        'usual': s.shift().expanding().median().values,
        'earlier_weeks': np.arange(len(s)),
    }))

# 4. One long table
weeks = pd.concat(parts, ignore_index=True)
weeks['ratio'] = weeks['total'] / weeks['usual']

weeks.head()

,Category,week,total,usual,earlier_weeks,ratio
0,Bills & Utilities,2025-11-03/2025-11-09,7024.03,NaN,0,NaN
1,Bills & Utilities,2025-11-10/2025-11-16,0.00,7024.030,1,0.0
2,Bills & Utilities,2025-11-17/2025-11-23,0.00,3512.015,2,0.0
3,Bills & Utilities,2025-11-24/2025-11-30,0.00,0.000,3,NaN
4,Bills & Utilities,2025-12-01/2025-12-07,6675.65,0.000,4,inf


Score it: a (week, category) is a real spree if it contains any `spree` row. Flag `ratio >= 3`, `total >= 500`, `earlier_weeks >= 4`, `usual > 0`.

In [25]:
# TODO
# Truth: (Category, week) pairs that contain at least one spree purchase
spree_keys = (
    exp.loc[exp['Anomaly'] == 'spree', ['Category', 'week']]
       .drop_duplicates()
       .assign(is_spree=True)
)
weeks = weeks.drop(columns='is_spree', errors='ignore').merge(
    spree_keys, on=['Category', 'week'], how='left')
weeks['is_spree'] = weeks['is_spree'].fillna(False).astype(bool)

# Rule
spree_pred = (
    (weeks['ratio'] >= 3) &
    (weeks['total'] >= 500) &
    (weeks['earlier_weeks'] >= 4) &
    (weeks['usual'] > 0)
)

# Score (manual: score() is built for per-purchase rows in exp)
tp = (spree_pred & weeks['is_spree']).sum()
fp = (spree_pred & ~weeks['is_spree']).sum()
fn = (~spree_pred & weeks['is_spree']).sum()

precision = tp / (tp + fp) if tp + fp else 0.0
recall    = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

print(f'precision {precision:.2f}  recall {recall:.2f}  f1 {f1:.2f}  '
      f'alerts {int(spree_pred.sum())}  real sprees {int(weeks["is_spree"].sum())}')

precision 0.23  recall 0.83  f1 0.36  alerts 22  real sprees 6


---
## E. Decide
**Gate:** if the ratio rule's F1 is more than **0.15** below Isolation Forest's, stop and tell Claude. We'd rethink the approach before building the app.

Otherwise, write your four numbers and one sentence for each saying why. These go into the app:
- `MULTIPLIER`
- `MIN_AMOUNT`
- `MIN_PURCHASES`
- `MIN_WEEKS`

**Also decide:** what should the app do about the false alarms you found in C? For example: leave them, skip that category, or compare a purchase with earlier purchases from the *same merchant* when there are some. Try your idea and check its F1.

In [26]:
week_rule = lambda mult, mw: ((weeks['ratio'] >= mult) & (weeks['total'] >= 500) &
                              (weeks['earlier_weeks'] >= mw) & (weeks['usual'] > 0))

wfa = weeks[week_rule(3, 4) & ~weeks['is_spree']]
print(wfa['Category'].value_counts())
wfa.sort_values('Category')[['Category', 'week', 'total', 'usual', 'ratio']]

Category
Entertainment    13
Shopping          4
Name: count, dtype: int64


,Category,week,total,usual,ratio
116,Entertainment,2026-01-19/2026-01-25,2080.35,105.590,19.702150
154,Entertainment,2026-10-12/2026-10-18,1208.03,250.335,4.825654
148,Entertainment,2026-08-31/2026-09-06,1908.16,311.295,6.129748
145,Entertainment,2026-08-10/2026-08-16,2508.38,333.100,7.530411
143,Entertainment,2026-07-27/2026-08-02,1063.08,333.100,3.191474
137,Entertainment,2026-06-15/2026-06-21,2418.55,333.100,7.260733
138,Entertainment,2026-06-22/2026-06-28,1197.42,377.270,3.173907
129,Entertainment,2026-04-20/2026-04-26,1938.59,333.100,5.819844
122,Entertainment,2026-03-02/2026-03-08,1992.30,655.545,3.039151
121,Entertainment,2026-02-23/2026-03-01,1853.92,333.100,5.565656


In [27]:
def wscore(pred):
    t = weeks['is_spree']
    tp, fp, fn = (pred & t).sum(), (pred & ~t).sum(), (~pred & t).sum()
    p = tp/(tp+fp) if tp+fp else 0; r = tp/(tp+fn) if tp+fn else 0
    return {'precision': p, 'recall': r, 'f1': 2*p*r/(p+r) if p+r else 0,
            'alerts': int(pred.sum()), 'caught': int(tp)}

pd.DataFrame({f'{m}×': wscore(week_rule(m, 4)) for m in [3, 4, 5]}).T.round(2)

,precision,recall,f1,alerts,caught
3×,0.23,0.83,0.36,22.0,5.0
4×,0.25,0.67,0.36,16.0,4.0
5×,0.29,0.67,0.40,14.0,4.0


In [28]:
fa5 = exp[(exp['ratio'] >= 5) & (exp['Amount'] >= 500) & (exp['earlier'] >= 5)
          & (exp['Anomaly'] != 'spike')]
fa5[['Date', 'Category', 'Description', 'Amount', 'usual', 'ratio']]

,Date,Category,Description,Amount,usual,ratio
302,2026-01-05 20:14:00+00:00,Bills & Utilities,Meralco,3353.71,608.99,5.507003


In [29]:
pd.DataFrame({f'{mw} weeks': wscore(week_rule(3, mw)) for mw in [4, 6]}).T.round(2)

,precision,recall,f1,alerts,caught
4 weeks,0.23,0.83,0.36,22.0,5.0
6 weeks,0.19,0.67,0.30,21.0,4.0


In [30]:
pd.DataFrame({
    '5× all':             wscore(week_rule(5, 4)),
    '5× no Entertainment': wscore(week_rule(5, 4) & (weeks['Category'] != 'Entertainment')),
}).T.round(2)

,precision,recall,f1,alerts,caught
5× all,0.29,0.67,0.4,14.0,4.0
5× no Entertainment,0.75,0.50,0.6,4.0,3.0


In [31]:
exp['m_earlier'] = exp.groupby(['Category', 'Description']).cumcount()
exp['m_usual'] = exp.groupby(['Category', 'Description'])['Amount'].transform(
    lambda s: s.shift().expanding().median())
exp['usual_m'] = np.where(exp['m_earlier'] >= 3, exp['m_usual'], exp['usual'])
exp['ratio_m'] = exp['Amount'] / exp['usual_m']

rows = {}
for m in [3, 4, 5]:
    keep = (exp['Amount'] >= 500) & (exp['earlier'] >= 5)
    rows[f'{m}× category'] = score((exp['ratio']   >= m) & keep)
    rows[f'{m}× merchant'] = score((exp['ratio_m'] >= m) & keep)
pd.DataFrame(rows).T.round(2)

,precision,recall,f1
3× category,0.45,0.77,0.57
3× merchant,0.33,0.77,0.47
4× category,0.64,0.69,0.67
4× merchant,0.60,0.69,0.64
5× category,0.88,0.54,0.67
5× merchant,0.67,0.62,0.64


In [33]:
# Shopping weekly false alarms: is each one explained by a single spike?
shop = wfa[wfa['Category'] == 'Shopping'][['Category', 'week', 'total', 'usual']]

spikes = (exp[exp['Anomaly'] == 'spike']
          .groupby(['Category', 'week'])['Amount'].sum()
          .rename('spike_amount').reset_index())

shop = shop.merge(spikes, on=['Category', 'week'], how='left').fillna({'spike_amount': 0})
shop['has_spike'] = shop['spike_amount'] > 0
shop['total_without_spike'] = shop['total'] - shop['spike_amount']
shop['ratio_without_spike'] = shop['total_without_spike'] / shop['usual']

print(shop.round({'total': 2, 'usual': 2, 'spike_amount': 2,
                  'total_without_spike': 2, 'ratio_without_spike': 2}).to_string())

   Category                   week     total    usual  spike_amount  has_spike  total_without_spike  ratio_without_spike
0  Shopping  2026-01-12/2026-01-18  10534.94  3042.04          0.00      False             10534.94                 3.46
1  Shopping  2026-07-06/2026-07-12  11025.22  2743.30          0.00      False             11025.22                 4.02
2  Shopping  2026-07-20/2026-07-26  20719.41  2874.77      16642.21       True              4077.20                 1.42
3  Shopping  2026-09-21/2026-09-27  11267.42  2844.23          0.00      False             11267.42                 3.96


My numbers:
- MULTIPLIER = 5
- MIN_AMOUNT = 500
- MIN_PURCHASES = 5
- MIN_WEEKS = 4

Why:
- MULTIPLIER = 5: it tied 4× on F1 (0.67) for single purchases, but 7 of its 8 alerts were real (precision 0.88 vs 0.64); for weeks it gives F1 0.40 vs 0.36 at 3×, so one multiplier works for both.
- MIN_AMOUNT = 500: it changed nothing on this data because every 4×+ purchase was over ₱1,000, but it keeps tiny categories from alerting on trivial amounts.
- MIN_PURCHASES = 5: not tuned; it keeps a category quiet until its median rests on more than one or two purchases.
- MIN_WEEKS = 4: raising it to 6 lowered F1 (0.36 → 0.30) and missed real sprees, because the weekly false alarms come from Entertainment's uneven spending, not from short history.

False alarms:
- Single purchases: leave them. At 5× only 1 false alarm remains (a Meralco bill), and comparing with the same merchant lowered F1 at 3×, 4× and 5× (cell 31).
- Weeks: 13 of the 17 weekly false alarms are Entertainment, so the app skips Entertainment for weekly alerts (F1 0.40 → 0.60). The 4 Shopping ones are not all explained by a spike: only 2026-07-20 has one, and the double-alert fix removes it (1.42× without the spike); the other 3 are below 5× (3.46×, 4.02×, 3.96×), so they never fire.